# Notebook 07c — SCTS-v2 Trust Score on Canonical Shared Samples

**Purpose:** Compute per-sample trust scores combining calibration confidence, explanation stability, and conformal coverage. Extends the existing UNSW-only `07_scts_v2.ipynb` to all 3 datasets × 6 model variants using canonical samples from 04c/06v3/05c.

**What is SCTS-v2 (verbatim from existing 07_scts_v2):**

SHAP-Calibrated Trust Score, version 2. A per-alert score in [0, 100] that combines three orthogonal signals:

| Component | What it measures | Source |
|---|---|---|
| **c₁ — Calibration confidence** | Calibrated probability of predicted class | Pre-saved `_test_proba_calibrated.npy` (verified identical to refit hybrid calibrators) |
| **c₂ — Explanation stability** | Worst-case per-sample Jaccard top-10 under adversarial perturbation | `stability_v2_per_sample_jaccard.csv` from 05c |
| **c₃ — Conformal coverage** | Per-sample conformal score at α=0.05 | Computed inline using split-conformal |

**Combination formula:** SCTS-v2 = (c₁ · c₂ · c₃)^(1/3) · 100

Geometric mean penalises asymmetry: weak in any one component → low SCTS even if others are strong. eps=1e-6 used to avoid log(0).

**Methodology improvements over existing 07_scts_v2:**
1. **Canonical samples:** Uses the 1000 per-dataset shared indices from 04c. Same samples that 06v3 (Krishna) and 05c (stability) used. Eliminates the sample-alignment problem the existing 07 had.
2. **Cleaner conformal split:** All test samples NOT in canonical 1000 serve as the conformal calibration set; canonical 1000 are scored. No sample leakage. More data for the quantile estimate.
3. **All 3 datasets, all 6 model variants:** 18 models total instead of just NSL-or-UNSW.

**Time estimate:** ~5-10 minutes (pure CPU, all data preloaded).

**Outputs:**
- `results/tables/scts_v2_canonical.csv` (per-sample SCTS scores: 18,000 rows)
- `results/tables/scts_v2_per_class_summary.csv` (per-class mean SCTS, c₁, c₂, c₃)
- `results/tables/scts_v2_alpha_sensitivity.csv` (α ∈ {0.05, 0.10, 0.20})
- `results/tables/scts_v2_validation.csv` (SCTS quartile vs accuracy)
- `results/tables/scts_v2_summary.json` (aggregate stats, conformal thresholds)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'✓ Ready in: {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
VARIANTS = ['5class_cw', '5class_smote']
MODELS_PER_DATASET = [f'{a}_{v}' for v in VARIANTS for a in ARCHITECTURES]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
ALPHAS = [0.05, 0.10, 0.20]
ALPHA_PRIMARY = 0.05  # Used for the main SCTS computation
EPS = 1e-6  # Floor for geometric mean

print(f'Scope: {len(DATASETS)} datasets × {len(MODELS_PER_DATASET)} models = {len(DATASETS) * len(MODELS_PER_DATASET)} model-dataset cells')
print(f'Primary alpha for SCTS: {ALPHA_PRIMARY}')
print(f'Sensitivity alphas: {ALPHAS}')

## 2. Conformal threshold and per-sample c₃

In [ ]:
def split_conformal_threshold(calib_probs, y_calib, alpha):
    """Compute conformal threshold (Romano et al. 2019 quantile).
    Nonconformity score s_i = 1 - p_hat(y_i | x_i) using TRUE label.
    Threshold = ceil((n+1)(1-alpha)) / n quantile of scores.
    """
    n = len(y_calib)
    scores = 1.0 - calib_probs[np.arange(n), y_calib]
    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(q_level, 1.0)
    return float(np.quantile(scores, q_level))

def empirical_coverage(test_probs, y_test, thresh):
    """Fraction of test samples where 1 - p_hat(y_true) <= threshold."""
    n = len(y_test)
    scores = 1.0 - test_probs[np.arange(n), y_test]
    return float((scores <= thresh).mean())

def component_3(test_probs, y_pred, thresh):
    """Per-sample c3 in [0, 1].
    Uses PREDICTED class (not true — not available at deployment).
    s = 1 - p_hat(predicted_class)
    c3 = max(0, 1 - s/thresh): 1 if very in-distribution, 0 at threshold or beyond.
    """
    s = 1.0 - test_probs[np.arange(len(y_pred)), y_pred]
    c3 = np.clip(1.0 - s / thresh, 0.0, 1.0)
    return c3.astype(np.float32)

print('✓ Conformal helpers loaded')

## 3. Load per-sample Jaccard (c₂ source) and compute worst-case per sample

In [ ]:
# Load stability per-sample Jaccard (from 05c)
stab_path = Path(REPO) / 'results' / 'tables' / 'stability_v2_per_sample_jaccard.csv'
df_stab = pd.read_csv(stab_path)
print(f'Loaded {stab_path.name}: {len(df_stab)} rows')
print(f'Columns: {df_stab.columns.tolist()}')

# Pivot: for each (dataset, model, sample_position) take MIN Jaccard across 3 perturbations
worst_case = df_stab.groupby(['dataset', 'model', 'sample_position'])['jaccard_top10'].min().reset_index()
worst_case.rename(columns={'jaccard_top10': 'worst_jaccard'}, inplace=True)
print(f'\nWorst-case per sample: {len(worst_case)} rows (expect 18 models × 1000 samples = 18000)')
print(f'Mean worst-case Jaccard: {worst_case["worst_jaccard"].mean():.3f}')
print(f'Min: {worst_case["worst_jaccard"].min():.3f}, Max: {worst_case["worst_jaccard"].max():.3f}')

# Build a lookup: (ds, model) -> array of shape (1000,) of worst-case Jaccards
c2_lookup = {}
for (ds, m), g in worst_case.groupby(['dataset', 'model']):
    g_sorted = g.sort_values('sample_position')
    c2_lookup[(ds, m)] = g_sorted['worst_jaccard'].values.astype(np.float32)

print(f'\nc2_lookup populated for {len(c2_lookup)} (dataset, model) combinations')

## 4. Main SCTS computation loop

In [ ]:
scts_records = []
alpha_records = []
conformal_meta = {}
validation_records = []
per_class_records = []

t_overall = time.time()
print(f'\n{"="*70}')
print(f'SCTS-v2 computation — {datetime.now().strftime("%H:%M:%S")}')
print(f'{"="*70}\n')

for ds in DATASETS:
    print(f'\n=== {ds} ===')
    
    # Load canonical indices
    canonical_eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    y_test_full = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    
    # Build conformal calibration mask: all test samples EXCEPT canonical 1000
    n_test = len(y_test_full)
    conformal_calib_mask = np.ones(n_test, dtype=bool)
    conformal_calib_mask[canonical_eval_idx] = False
    n_conformal_calib = conformal_calib_mask.sum()
    
    y_canonical = y_test_full[canonical_eval_idx]
    y_conformal_calib = y_test_full[conformal_calib_mask]
    
    print(f'  n_test={n_test}, n_conformal_calib={n_conformal_calib}, n_canonical={len(canonical_eval_idx)}')
    
    for model_name in MODELS_PER_DATASET:
        # Load pre-saved calibrated probabilities (verified identical to refit hybrid calibrator output)
        prob_path = Path(REPO) / 'calibrators' / ds / f'{model_name}_test_proba_calibrated.npy'
        if not prob_path.exists():
            print(f'  ❌ {model_name}: calibrated probs missing')
            continue
        calibrated_proba_full = np.load(prob_path)
        
        # Split into conformal calib and canonical (test for SCTS)
        calibrated_proba_conformal = calibrated_proba_full[conformal_calib_mask]
        calibrated_proba_canonical = calibrated_proba_full[canonical_eval_idx]
        
        # ===== c1: calibration confidence on canonical samples =====
        y_pred_canonical = calibrated_proba_canonical.argmax(axis=1)
        c1 = calibrated_proba_canonical[np.arange(len(y_pred_canonical)), y_pred_canonical].astype(np.float32)
        
        # ===== c2: worst-case Jaccard (from stability_v2_per_sample_jaccard.csv) =====
        if (ds, model_name) not in c2_lookup:
            print(f'  ⚠ {model_name}: no c2 data, skipping')
            continue
        c2 = c2_lookup[(ds, model_name)]  # already (1000,) float32
        
        # ===== c3: conformal at primary alpha =====
        thresh_primary = split_conformal_threshold(
            calibrated_proba_conformal, y_conformal_calib, ALPHA_PRIMARY
        )
        c3 = component_3(calibrated_proba_canonical, y_pred_canonical, thresh_primary)
        
        # Empirical coverage on canonical 1000 (uses TRUE labels)
        emp_coverage = empirical_coverage(calibrated_proba_canonical, y_canonical, thresh_primary)
        
        # ===== SCTS-v2 =====
        geo_mean = (np.clip(c1, EPS, 1.0) * np.clip(c2, EPS, 1.0) * np.clip(c3, EPS, 1.0)) ** (1/3)
        scts = (geo_mean * 100).astype(np.float32)
        
        # ===== Accuracy on canonical (for validation) =====
        correct = (y_pred_canonical == y_canonical).astype(float)
        
        # ===== Per-sample records =====
        for i in range(len(scts)):
            scts_records.append({
                'dataset': ds, 'model': model_name,
                'sample_position': i,
                'true_class': int(y_canonical[i]),
                'pred_class': int(y_pred_canonical[i]),
                'correct': int(correct[i]),
                'c1': float(c1[i]),
                'c2': float(c2[i]),
                'c3': float(c3[i]),
                'scts': float(scts[i]),
            })
        
        # ===== Conformal metadata =====
        conformal_meta[f'{ds}/{model_name}'] = {
            'threshold_alpha_0.05': float(thresh_primary),
            'empirical_coverage_on_canonical': float(emp_coverage),
            'n_conformal_calibration_samples': int(n_conformal_calib),
        }
        
        # ===== Alpha sensitivity =====
        for alpha in ALPHAS:
            thresh_a = split_conformal_threshold(calibrated_proba_conformal, y_conformal_calib, alpha)
            c3_a = component_3(calibrated_proba_canonical, y_pred_canonical, thresh_a)
            geo_a = (np.clip(c1, EPS, 1.0) * np.clip(c2, EPS, 1.0) * np.clip(c3_a, EPS, 1.0)) ** (1/3)
            scts_a = geo_a * 100
            emp_cov_a = empirical_coverage(calibrated_proba_canonical, y_canonical, thresh_a)
            alpha_records.append({
                'dataset': ds, 'model': model_name, 'alpha': alpha,
                'threshold': float(thresh_a),
                'empirical_coverage': float(emp_cov_a),
                'mean_scts': float(scts_a.mean()),
                'median_scts': float(np.median(scts_a)),
            })
        
        # ===== SCTS quartile validation =====
        overall_acc = correct.mean()
        pearson_corr = np.corrcoef(scts, correct)[0, 1] if scts.std() > 1e-9 else 0.0
        for lo, hi in [(0, 25), (25, 50), (50, 75), (75, 101)]:
            mask = (scts >= lo) & (scts < hi)
            n = int(mask.sum())
            acc = float(correct[mask].mean()) if n > 0 else float('nan')
            validation_records.append({
                'dataset': ds, 'model': model_name,
                'scts_bin_low': lo, 'scts_bin_high': hi,
                'n': n, 'accuracy': acc,
                'overall_accuracy': float(overall_acc),
                'pearson_corr_scts_correct': float(pearson_corr),
            })
        
        # ===== Per-class summary =====
        for c, cname in enumerate(CLASS_NAMES_5):
            mask = y_canonical == c
            if mask.sum() > 0:
                per_class_records.append({
                    'dataset': ds, 'model': model_name,
                    'true_class': cname, 'class_idx': c,
                    'n': int(mask.sum()),
                    'mean_scts': float(scts[mask].mean()),
                    'mean_c1': float(c1[mask].mean()),
                    'mean_c2': float(c2[mask].mean()),
                    'mean_c3': float(c3[mask].mean()),
                    'accuracy': float(correct[mask].mean()),
                })
        
        print(f'  ▶ {model_name:<22} thresh={thresh_primary:.3f}, emp_cov={emp_coverage:.3f}, '
              f'mean SCTS={scts.mean():.1f}, median={np.median(scts):.1f}, '
              f'acc={overall_acc:.3f}, corr={pearson_corr:+.3f}')

elapsed = (time.time() - t_overall) / 60
print(f'\nTotal time: {elapsed:.1f} min')
print(f'Per-sample records: {len(scts_records)}')
print(f'Per-class records: {len(per_class_records)}')
print(f'Alpha sensitivity records: {len(alpha_records)}')

## 5. Build DataFrames and save CSVs

In [ ]:
df_scts = pd.DataFrame(scts_records)
df_perclass = pd.DataFrame(per_class_records)
df_alpha = pd.DataFrame(alpha_records)
df_validation = pd.DataFrame(validation_records)

out_dir = Path(REPO) / 'results' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)

df_scts.to_csv(out_dir / 'scts_v2_canonical.csv', index=False)
df_perclass.to_csv(out_dir / 'scts_v2_per_class_summary.csv', index=False)
df_alpha.to_csv(out_dir / 'scts_v2_alpha_sensitivity.csv', index=False)
df_validation.to_csv(out_dir / 'scts_v2_validation.csv', index=False)

print(f'✓ Saved 4 CSVs:')
print(f'  scts_v2_canonical.csv ({len(df_scts)} rows)')
print(f'  scts_v2_per_class_summary.csv ({len(df_perclass)} rows)')
print(f'  scts_v2_alpha_sensitivity.csv ({len(df_alpha)} rows)')
print(f'  scts_v2_validation.csv ({len(df_validation)} rows)')

## 6. Print headline findings

In [ ]:
print('=' * 70)
print('SCTS-v2 HEADLINE FINDINGS')
print('=' * 70)

# Aggregate by (dataset, model)
df_scts['architecture'] = df_scts['model'].apply(
    lambda m: 'rf' if 'rf' in m else ('xgb' if 'xgb' in m else 'dnn')
)
df_scts['variant'] = df_scts['model'].apply(
    lambda m: 'cw' if 'cw' in m else 'smote'
)

print('\n--- Mean SCTS per (dataset, architecture) averaged across variants ---')
for ds in DATASETS:
    sub = df_scts[df_scts['dataset'] == ds]
    pivot = sub.groupby('architecture')['scts'].mean().round(1)
    print(f'  {ds:<18} {dict(pivot)}')

print('\n--- Validation: SCTS quartile vs accuracy ---')
print('(If SCTS is meaningful, higher SCTS bin should have higher accuracy)')
agg_val = df_validation.groupby(['scts_bin_low', 'scts_bin_high']).agg({
    'n': 'sum',
    'accuracy': 'mean',
}).reset_index()
for _, row in agg_val.iterrows():
    print(f'  SCTS [{int(row["scts_bin_low"]):>3},{int(row["scts_bin_high"]):>3}): '
          f'n={int(row["n"]):>5}, mean_acc={row["accuracy"]:.3f}')

print('\n--- Pearson correlation SCTS vs correctness ---')
corr_summary = df_validation.groupby(['dataset', 'model'])['pearson_corr_scts_correct'].first()
print(f'  Mean correlation across 18 models: {corr_summary.mean():+.3f}')
print(f'  Min: {corr_summary.min():+.3f}, Max: {corr_summary.max():+.3f}')
print(f'  Models with corr > 0: {(corr_summary > 0).sum()}/{len(corr_summary)}')

print('\n--- Alpha sensitivity (mean SCTS across all 18 models) ---')
alpha_pivot = df_alpha.groupby('alpha')[['threshold', 'empirical_coverage', 'mean_scts']].mean().round(3)
print(alpha_pivot)

print('\n--- Per-class SCTS (mean across 18 models, by true class) ---')
perclass_pivot = df_perclass.groupby('true_class').agg({
    'mean_scts': 'mean', 'mean_c1': 'mean', 'mean_c2': 'mean', 'mean_c3': 'mean',
    'accuracy': 'mean', 'n': 'sum',
}).round(3)
print(perclass_pivot.reindex(CLASS_NAMES_5))

## 7. Save summary JSON

In [ ]:
def to_json_safe(obj):
    """Recursive numpy/tuple-key sanitizer (proven across 06v3 and 05c)."""
    if isinstance(obj, dict):
        new_dict = {}
        for k, v in obj.items():
            if isinstance(k, tuple):
                k = '|'.join(str(x) for x in k)
            elif not isinstance(k, (str, int, float, bool)) and k is not None:
                k = str(k)
            new_dict[k] = to_json_safe(v)
        return new_dict
    elif isinstance(obj, list):
        return [to_json_safe(x) for x in obj]
    elif isinstance(obj, (np.bool_, np.generic)):
        return obj.item()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

summary = {
    'timestamp': datetime.now().isoformat(),
    'method': 'canonical_shared_samples_per_dataset',
    'n_datasets': len(DATASETS),
    'n_models': len(MODELS_PER_DATASET) * len(DATASETS),
    'n_samples_per_model': 1000,
    'primary_alpha': ALPHA_PRIMARY,
    'sensitivity_alphas': ALPHAS,
    'overall_stats': {
        'mean_scts': float(df_scts['scts'].mean()),
        'median_scts': float(df_scts['scts'].median()),
        'min_scts': float(df_scts['scts'].min()),
        'max_scts': float(df_scts['scts'].max()),
        'std_scts': float(df_scts['scts'].std()),
    },
    'mean_components': {
        'c1_calibration': float(df_scts['c1'].mean()),
        'c2_stability': float(df_scts['c2'].mean()),
        'c3_conformal': float(df_scts['c3'].mean()),
    },
    'mean_scts_by_dataset_architecture': df_scts.groupby(['dataset', 'architecture'])['scts'].mean().to_dict(),
    'mean_pearson_corr_scts_correctness': float(
        df_validation.groupby(['dataset', 'model'])['pearson_corr_scts_correct'].first().mean()
    ),
    'conformal_thresholds': conformal_meta,
    'alpha_sensitivity_mean': df_alpha.groupby('alpha')[['threshold', 'empirical_coverage', 'mean_scts']].mean().to_dict(),
}

summary_safe = to_json_safe(summary)
out_path = Path(REPO) / 'results' / 'tables' / 'scts_v2_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary_safe, f, indent=2)
print(f'✓ Saved: scts_v2_summary.json')

print('\n=== Summary contents ===')
print(json.dumps(summary_safe, indent=2)[:3000])
print('... (truncated)')

## 8. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/07c_scts_canonical.ipynb
!git add results/tables/scts_v2_canonical.csv
!git add results/tables/scts_v2_per_class_summary.csv
!git add results/tables/scts_v2_alpha_sensitivity.csv
!git add results/tables/scts_v2_validation.csv
!git add results/tables/scts_v2_summary.json
!git status --short
!git commit -m 'Notebook 07c: SCTS-v2 trust score on canonical samples (18 models, geometric mean of calibration+stability+conformal, alpha sensitivity, per-class summary)'
!git push origin main